# SimSat DiLoCo Round 0 Learner — v4 (diagnostic)

v3 result: continued adapter STILL bit-identical to seed even after `inference_mode: false` patch. eval_loss_before == eval_loss_after to 14 decimals, so the model is genuinely unchanged at eval time even though forward+backward+optimizer.step ran for 120 steps with 22.8M trainable params.

v4 strategy: bypass the runner's subprocess and call its functions inline so we can:
1. Snapshot LoRA weight checksums BEFORE training (init)
2. Snapshot AFTER training but BEFORE save (in-memory)
3. Snapshot AFTER save (on-disk)

Three outcomes, three diagnoses:
- init == in-memory == on-disk : optimizer never updated LoRA params (gradient flow broken)
- init != in-memory but saved == seed : optimizer worked, save_pretrained writes stale weights
- init != in-memory == saved : training works, our prior measurements were wrong

Also adds `prepare_model_for_kbit_training(base)` BEFORE adapter attach as the leading hypothesis fix (the runner skips this, which can break gradient flow under bnb 4-bit + gradient checkpointing).

In [ ]:
import hashlib, json, os, shutil, sys
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')

# --- Resolve source bundle (auto-extracted by Kaggle) ---
src_marker = next(INPUT.rglob('continue_gemma4_adapter.py'))
SRC = src_marker.parents[1]  # bundle root (parent of kaggle/)
print(f'SRC: {SRC}')

# --- Stage writable adapter + flip inference_mode ---
READ_ONLY = next(INPUT.rglob('adapter_config.json')).parent
ADAPTER = WORK / 'global_adapter_writable'
if ADAPTER.exists():
    shutil.rmtree(ADAPTER)
shutil.copytree(READ_ONLY, ADAPTER)
cfg_path = ADAPTER / 'adapter_config.json'
cfg = json.loads(cfg_path.read_text())
prev_mode = cfg.get('inference_mode')
cfg['inference_mode'] = False
cfg_path.write_text(json.dumps(cfg, indent=2))
print(f'Adapter staged: {ADAPTER}  (inference_mode: {prev_mode!r} -> False)')

def _sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

SEED_HASH = _sha256(ADAPTER / 'adapter_model.safetensors')
print(f'Seed sha256: {SEED_HASH}')

# --- Make the runner importable ---
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
sys.path.insert(0, str(SRC / 'kaggle'))
sys.path.insert(0, str(SRC))

import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'unsloth', 'unsloth_zoo', 'torchao'], check=False)
shutil.rmtree('/kaggle/working/unsloth_compiled_cache', ignore_errors=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers>=4.51.0', 'peft>=0.12.0', 'accelerate>=0.33.0',
    'bitsandbytes>=0.44.0', 'datasets>=2.19.0', 'safetensors>=0.4.3'])

import continue_gemma4_adapter as runner
import torch
from transformers import AutoTokenizer
from peft import prepare_model_for_kbit_training

# --- Load model + attach adapter ---
MODEL_ID = '/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1'
TRAIN_PATH = next(Path('/kaggle/input').rglob('simsat_train.jsonl'))
print(f'MODEL: {MODEL_ID}')
print(f'TRAIN: {TRAIN_PATH}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

rows = runner._load_jsonl(str(TRAIN_PATH))
dataset = runner.ChatMLDataset(rows, tokenizer, 1024)
print(f'dataset: {len(dataset)} rows | tokens={dataset.total_tokens:,} supervised={dataset.supervised_tokens:,}')

base = runner._load_base_model(MODEL_ID)
# v4 fix: prepare_model_for_kbit_training BEFORE adapter attach
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False})
print('base prepared with prepare_model_for_kbit_training (use_gradient_checkpointing=True, use_reentrant=False)')
model = runner._attach_trainable_adapter(base, str(ADAPTER))

# --- Snapshot 1: LoRA weights at INIT ---
def _lora_state(model):
    """Return a dict of name -> (sha256 of tensor bytes, sum of abs values)"""
    out = {}
    for name, param in model.named_parameters():
        if 'lora_' not in name:
            continue
        t = param.detach().cpu().float().numpy()
        h = hashlib.sha256(t.tobytes()).hexdigest()
        out[name] = (h, float(abs(t).sum()))
    return out

INIT = _lora_state(model)
print(f'INIT lora params: {len(INIT)}')
sample_name = next(iter(INIT))
print(f'  sample param: {sample_name}  hash={INIT[sample_name][0][:16]}  abs_sum={INIT[sample_name][1]:.4f}')

# --- Train ---
pad_id = int(tokenizer.pad_token_id or tokenizer.eos_token_id)
loss_before = runner._evaluate_loss(model, dataset, pad_id, 16)
print(f'loss_before: {loss_before}')

steps = runner._train(model, dataset, pad_token_id=pad_id, batch_size=1, grad_accum=8,
                       max_steps=120, epochs=1, lr=5e-5, warmup_steps=10, seed=42)
loss_after = runner._evaluate_loss(model, dataset, pad_id, 16)
print(f'loss_after: {loss_after}')

# --- Snapshot 2: LoRA weights AFTER training (in memory) ---
POST = _lora_state(model)
changed = sum(1 for k in INIT if INIT[k][0] != POST[k][0])
print(f'IN-MEMORY: {changed}/{len(INIT)} lora params changed')
print(f'  sample param: {sample_name}  hash={POST[sample_name][0][:16]}  abs_sum={POST[sample_name][1]:.4f}')

# --- Save ---
OUT = WORK / 'diloco_continued_adapter'
OUT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUT))
tokenizer.save_pretrained(str(OUT))
OUT_HASH = _sha256(OUT / 'adapter_model.safetensors')
print(f'Continued sha256: {OUT_HASH}')

# --- Snapshot 3: LoRA weights from saved file ---
from safetensors import safe_open
SAVED = {}
with safe_open(OUT / 'adapter_model.safetensors', framework='numpy') as f:
    for k in f.keys():
        if 'lora_' in k:
            t = f.get_tensor(k)
            SAVED[k] = (hashlib.sha256(t.tobytes()).hexdigest(), float(abs(t).sum()))
print(f'SAVED lora params: {len(SAVED)}')

# Find the saved equivalent of our sample (peft sometimes prefixes with 'base_model.model.')
saved_match = None
stripped = sample_name.replace('base_model.model.', '')
for k in SAVED:
    if k == sample_name or k == stripped or sample_name.endswith(k) or k.endswith(stripped):
        saved_match = k
        break
if saved_match:
    print(f'  sample param: {saved_match}  hash={SAVED[saved_match][0][:16]}  abs_sum={SAVED[saved_match][1]:.4f}')

# --- Diagnosis ---
print()
print('=== DIAGNOSIS ===')
print(f'INIT == POST (in-mem): {sum(1 for k in INIT if INIT[k][0] == POST[k][0])}/{len(INIT)} params unchanged')
print(f'Continued sha256 == seed sha256: {OUT_HASH == SEED_HASH}')
if OUT_HASH == SEED_HASH and changed > 0:
    print('VERDICT: Training updated LoRA in-memory but save_pretrained wrote stale weights.')
elif OUT_HASH == SEED_HASH and changed == 0:
    print('VERDICT: Optimizer did not update LoRA at all (gradient flow broken).')
else:
    print('VERDICT: Training + save both worked. Continued adapter differs from seed.')

# --- Run the export step manually ---
from diloco_lab.learner_export import export_update, _parse_fragment_ids
OUTBOX = WORK / 'diloco_outbox'
update_dir = export_update(
    base_adapter=str(ADAPTER),
    trained_adapter=str(OUT),
    outbox=str(OUTBOX),
    experiment='gemma4good_simsat_arc3_foundation',
    round_id=0,
    learner_id='kaggle-t4-simsat-round0-a',
    project='simsat',
    dataset_id='simsat-gemma4-v3-reviewed',
    tokens=dataset.total_tokens,
    steps=max(1, steps),
    num_fragments=8,
    fragment_ids=_parse_fragment_ids('all', 8),
    loss_before=loss_before,
    loss_after=loss_after,
    metrics={'supervised_tokens': dataset.supervised_tokens},
)
summary = {
    'learner_id': 'kaggle-t4-simsat-round0-a',
    'project': 'simsat',
    'dataset_id': 'simsat-gemma4-v3-reviewed',
    'train_path': str(TRAIN_PATH),
    'base_adapter': str(ADAPTER),
    'output_dir': str(OUT),
    'round_id': 0,
    'steps': steps,
    'total_tokens': dataset.total_tokens,
    'supervised_tokens': dataset.supervised_tokens,
    'loss_before': loss_before,
    'loss_after': loss_after,
    'in_memory_changed_params': changed,
    'seed_sha256': SEED_HASH,
    'continued_sha256': OUT_HASH,
    'diloco_update_dir': str(update_dir),
}
(OUT / 'diloco_learner_summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True))
print(f'\nsummary: {OUT / "diloco_learner_summary.json"}')